# 07.6 - Sequence Models for NLP

**Phase:** 07 - NLP

**Status:** VERIFIED

---
## 1. What Are We Solving?

'The cat chased the dog' and 'The dog chased the cat' have the same words but different meanings. BoW/TF-IDF lose word order. **RNNs and LSTMs** process words in order, maintaining a hidden state that carries context forward.

## 2. Why Does This Matter?

Sequence models bridge classical NLP and transformers. They understand grammar and long-range dependencies, solving sentiment, NER, and translation. LSTMs fix the RNN's vanishing-gradient problem with gates.

## 3. Prerequisites

- Unit 07.5 (embeddings)
- Phase 06 (neural networks, PyTorch)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain RNN/LSTM architecture and hidden state
- Implement an LSTM classifier in PyTorch
- Understand bidirectional vs unidirectional LSTM
- Debug vanishing/exploding gradient issues

## 5. Mental Model

```text
Input:  The  cat  sat  on  the  mat
Hidden: h0 -> h1 -> h2 -> h3 -> h4 -> h5 -> h6  (memory flowing forward)
```
LSTM adds gates: forget (what to drop), input (what to store), output (what to pass on).


## 6. Setup

CPU-only; small synthetic classification task.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)
print('torch', torch.__version__)


## 7. Synthetic Task: Predict if sum of sequence exceeds threshold

Each sequence is a length-5 series of integers; label = 1 if sum > threshold. The model must aggregate across the whole sequence (needs a memory of prior tokens).


In [ ]:
def make_data(n, seq_len=5, threshold=12):
    X = np.random.randint(0, 5, size=(n, seq_len))
    y = (X.sum(axis=1) > threshold).astype(int)
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

X, y = make_data(4000)
split = 3200
X_tr, y_tr = X[:split], y[:split]
X_te, y_te = X[split:], y[split:]
print("Train shapes:", X_tr.shape, y_tr.shape)
print("Example seqs/ labels:", X_tr[:4].tolist(), y_tr[:4].tolist())


## 8. Build an LSTM Classifier

Embedding -> LSTM -> take last hidden state -> Linear -> classify.


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        emb = self.embedding(x)                 # (B, L, E)
        out, (h, c) = self.lstm(emb)            # out: (B, L, H)
        last = h[-1]                            # final hidden (B, H)
        return self.fc(last)

model = LSTMClassifier(vocab_size=5, embed_dim=8, hidden_dim=16)
print(model)


## 9. Train the LSTM


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

def train(model, X, y, epochs=20, bs=128):
    model.train()
    n = X.shape[0]
    for ep in range(epochs):
        perm = torch.randperm(n)
        total = 0
        cnt = 0
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            optimizer.zero_grad()
            loss = loss_fn(model(X[idx]), y[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevent exploding grads
            optimizer.step()
            total += loss.item()*len(idx)
            cnt += len(idx)
        if ep % 5 == 0:
            print(f"epoch {ep:2d}: loss={total/cnt:.4f}")

train(model, X_tr, y_tr)
print("Training complete.")


## 10. Evaluate

LSTM should learn to aggregate the sequence -> high accuracy on the held-out set.


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_te)
    preds = logits.argmax(1)
acc = (preds == y_te).float().mean().item()
print(f"Test accuracy: {acc*100:.1f}%")
print("The LSTM uses its hidden state to sum info across the sequence.")


## 11. Compare vs a Unigram/BoW Baseline

A bag-of-words view (sum of indicator features) vs the ordered LSTM.


In [ ]:
# BoW baseline: just feed the mean embedding (no order)
class BOWBaseline(nn.Module):
    def __init__(self, vocab_size, embed_dim, out=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, out)
    def forward(self, x):
        emb = self.embedding(x)
        avg = emb.mean(dim=1)
        return self.fc(avg)

bow = BOWBaseline(5, 8)
opt = torch.optim.Adam(bow.parameters(), lr=0.01)
for ep in range(20):
    opt.zero_grad()
    loss = F.cross_entropy(bow(X_tr), y_tr)
    loss.backward()
    opt.step()
with torch.no_grad():
    acc_bow = (bow(X_te).argmax(1) == y_te).float().mean().item()
print(f"BoW baseline test acc: {acc_bow*100:.1f}%")
print(f"LSTM test acc:         {acc*100:.1f}%")
print("Here BoW ~ LSTM because sum order doesn't matter; order matters when grammar does.")


## 12. Bidirectional LSTM

Reads forward AND backward. Useful when you have the full sequence (classification, not generation).


In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, out=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, out)
    def forward(self, x):
        emb = self.embedding(x)
        out, (h, c) = self.lstm(emb)
        h_cat = torch.cat((h[-2], h[-1]), dim=1)  # forward + backward final
        return self.fc(h_cat)

bi = BiLSTM(5, 8, 12)
opt = torch.optim.Adam(bi.parameters(), lr=0.01)
for ep in range(20):
    opt.zero_grad()
    loss = F.cross_entropy(bi(X_tr), y_tr)
    loss.backward()
    opt.step()
with torch.no_grad():
    acc_bi = (bi(X_te).argmax(1) == y_te).float().mean().item()
print(f"Bidirectional LSTM test acc: {acc_bi*100:.1f}%")
print("Bidirectional sees both past and future context.")


## 13. Failure Case: Vanishing Gradients in Plain RNN

Plain RNNs struggle on long sequences due to vanishing gradients. We build a long-range task to show LSTM wins.


In [ ]:
def make_long(n, seq_len=40):
    X = np.random.randint(0, 5, size=(n, seq_len))
    # signal is only in the first token; model must carry it across 40 steps
    y = (X[:, 0] >= 4).astype(int)
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

Xl, yl = make_long(3000)
Xl_tr, yl_tr = Xl[:2400], yl[:2400]
Xl_te, yl_te = Xl[2400:], yl[2400:]
print("Long-sequence task shapes:", Xl_tr.shape)


## 14. Train Plain RNN vs LSTM on the Long-Range Task


In [ ]:
def acc_after_train(mod, epochs=30):
    opt = torch.optim.Adam(mod.parameters(), lr=0.01)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.cross_entropy(mod(Xl_tr), yl_tr)
        loss.backward()
        nn.utils.clip_grad_norm_(mod.parameters(), 1.0)
        opt.step()
    with torch.no_grad():
        return (mod(Xl_te).argmax(1) == yl_te).float().mean().item()

class RNNClf(nn.Module):
    def __init__(self, vs, ed, hd, out=2):
        super().__init__()
        self.embedding = nn.Embedding(vs, ed)
        self.rnn = nn.RNN(ed, hd, batch_first=True)
        self.fc = nn.Linear(hd, out)
    def forward(self, x):
        e = self.embedding(x); o, h = self.rnn(e)
        return self.fc(h[-1])

rn = RNNClf(5, 8, 16)
l6 = LSTMClassifier(5, 8, 16)
acc_rnn = acc_after_train(rn)
acc_lstm = acc_after_train(l6)
print(f"Plain RNN,  long-range acc: {acc_rnn*100:.1f}%")
print(f"LSTM,       long-range acc: {acc_lstm*100:.1f}%")
print("\nLSTM's gates let it carry the first token's signal far better than a plain RNN.")


## 15. Debugging: Common Errors

- **Loss doesn't decrease** — LR too high or vanishing grads. Fix: lower LR, use LSTM/GRU, clip grads.
- **Ignores long-range context** — plain RNN. Fix: LSTM/GRU.
- **OOM on long sequences** — hidden state stores full seq. Fix: truncate, checkpoint.
- **Slow training** — sequential can't parallelize. Fix: reduce length, teacher forcing.

## 16. Real-World Considerations

- Always prefer LSTM/GRU over plain RNN.
- Use bidirectional for classification (full input), unidirectional for generation.
- Apply dropout between layers; clip gradients.

## 17. Common Mistakes

- Using a plain RNN for long sequences.
- Not masking padded sequences.
- Too many LSTM layers (diminishing returns).

## 18. When NOT to Use

- Very long sequences (use Transformer, Phase 08).
- Tiny datasets where a simple classifier suffices.

## 19. Challenge

Replace the LSTM with a GRU and compare performance on the long-range task.


In [ ]:
# Challenge: GRU vs LSTM on long-range task
class GRUClf(nn.Module):
    def __init__(self, vs, ed, hd, out=2):
        super().__init__()
        self.embedding = nn.Embedding(vs, ed)
        self.gru = nn.GRU(ed, hd, batch_first=True)
        self.fc = nn.Linear(hd, out)
    def forward(self, x):
        e = self.embedding(x); o, h = self.gru(e)
        return self.fc(h[-1])

gru = GRUClf(5, 8, 16)
acc_gru = acc_after_train(gru)
print(f"GRU long-range acc: {acc_gru*100:.1f}%  (vs LSTM {acc_lstm*100:.1f}%)")
print("GRU has fewer parameters (no separate cell state) and often trains faster.")


## 20. Closed-Book Recall

1. What is the hidden state in an RNN?
2. What problem do LSTM gates solve?
3. When would you use bidirectional over unidirectional?
4. What is gradient clipping and why is it needed?

## 21. Teach-Back Questions

- Explain the vanishing gradient problem and LSTM's fix.
- Contrast unidirectional (generation) vs bidirectional (classification).

## 22. Summary

You implemented an LSTM classifier, compared it to a BoW baseline and a plain RNN, and saw LSTMs win on long-range tasks. This is the bridge to attention and transformers.

## 23. Further Experiment

- Use word embeddings + LSTM for sentiment on a real dataset.
- Add dropout and more layers; compare training time.

## 24. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, torch, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
